# MouseBrain v2 (FIXED) — Orthogonal Dodecahedral Core + Quasicrystal Chained Tracts (Colab)

This notebook builds a synthetic **mouse-brain-like** 3D scaffold using:
- **Orthogonal dodecahedral tiling** via an **FCC / rhombic-dodecahedral honeycomb** core
- **Quasicrystalline chained lattices** via **Ammann–Beenker (8-fold) cut-and-project layers** stitched into the core

Then it runs **v2 dynamics**:
- **LIF spiking network** over the fused scaffold
- **Distance-based conduction delays**
- **Pair-based STDP** on plastic synapses
- Automated outputs + **bundle zip** (graphs, metrics, plots)

✅ Fix included: **GEXF export sanitizes numpy arrays** (NetworkX GEXF cannot serialize `numpy.ndarray` attributes).


In [ ]:
# Colab setup
import os, sys, math, random, time, json, zipfile
import numpy as np

try:
    import networkx as nx
    import matplotlib.pyplot as plt
except Exception:
    !pip -q install networkx matplotlib
    import networkx as nx
    import matplotlib.pyplot as plt

from mpl_toolkits.mplot3d import Axes3D  # noqa
print("Ready:", "nx", nx.__version__)


In [ ]:
\
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import math, random, os, json, zipfile, time

# ============================================================
# Utilities
# ============================================================

def set_seed(seed: int = 7):
    random.seed(seed)
    np.random.seed(seed)

def unit(v):
    n = np.linalg.norm(v)
    return v / (n + 1e-12)

def rot_matrix(axis, theta):
    axis = unit(np.array(axis, dtype=float))
    a = math.cos(theta / 2.0)
    b, c, d = -axis * math.sin(theta / 2.0)
    aa, bb, cc, dd = a*a, b*b, c*c, d*d
    bc, ad, ac, ab, bd, cd = b*c, a*d, a*c, a*b, b*d, c*d
    return np.array([
        [aa + bb - cc - dd, 2*(bc + ad),       2*(bd - ac)],
        [2*(bc - ad),       aa + cc - bb - dd, 2*(cd + ab)],
        [2*(bd + ac),       2*(cd - ab),       aa + dd - bb - cc]
    ], dtype=float)

def gaussian_prob(d, sigma):
    return math.exp(-(d*d)/(2*sigma*sigma + 1e-12))

def k_nearest(points, k=12):
    P = np.asarray(points)
    N = P.shape[0]
    out = []
    for i in range(N):
        d2 = np.sum((P - P[i])**2, axis=1)
        idx = np.argsort(d2)
        out.append([int(x) for x in idx[1:k+1]])
    return out

# ============================================================
# 1) Orthogonal dodecahedral tiling scaffold (FCC -> rhombic dodecahedra Voronoi)
# ============================================================

def generate_fcc_points(bounds, a=1.0):
    B = float(bounds)
    basis = np.array([
        [0.0, 0.0, 0.0],
        [0.0, 0.5, 0.5],
        [0.5, 0.0, 0.5],
        [0.5, 0.5, 0.0],
    ]) * a
    n = int(math.ceil((B / a) + 2))
    pts = []
    for i in range(-n, n+1):
        for j in range(-n, n+1):
            for k in range(-n, n+1):
                cell = np.array([i, j, k], dtype=float) * a
                for b in basis:
                    p = cell + b
                    if np.all(np.abs(p) <= B):
                        pts.append(p)
    pts = np.unique(np.round(np.array(pts), 6), axis=0)
    return pts

def build_dodecahedral_graph(points, nn_k=14, shell_factor=1.18):
    P = np.asarray(points)
    N = P.shape[0]
    knn = k_nearest(P, k=nn_k)
    first_d = []
    for i in range(N):
        j = knn[i][0]
        first_d.append(np.linalg.norm(P[i] - P[j]))
    d0 = float(np.median(first_d))
    thresh = d0 * shell_factor

    G = nx.Graph()
    for i in range(N):
        G.add_node(i, pos=P[i])
    for i in range(N):
        for j in knn[i]:
            d = float(np.linalg.norm(P[i] - P[j]))
            if d <= thresh:
                G.add_edge(i, j, kind="dodeca", w=1.0/(d+1e-9), dist=d)
    return G

# ============================================================
# 2) Quasicrystal chained lattices (Ammann–Beenker cut-and-project layers)
# ============================================================

def ammann_beenker_points_2d(n_pts=600, seed=7, radius=18.0):
    set_seed(seed)
    m = int(max(4, round((n_pts ** 0.25) * 6)))
    candidates = [(a,b,c,d) for a in range(-m,m+1)
                          for b in range(-m,m+1)
                          for c in range(-m,m+1)
                          for d in range(-m,m+1)]
    random.shuffle(candidates)
    angles = [0.0, math.pi/4, math.pi/2, 3*math.pi/4]
    Bp = np.array([[math.cos(t), math.sin(t)] for t in angles], dtype=float)  # 4x2
    angles_i = [t + math.pi/8 for t in angles]
    Bi = np.array([[math.cos(t), math.sin(t)] for t in angles_i], dtype=float)
    pts = []
    win_r = 1.25
    for (a,b,c,d) in candidates:
        v = np.array([a,b,c,d], dtype=float)
        p = v @ Bp
        q = v @ Bi
        if np.linalg.norm(q) <= win_r and np.linalg.norm(p) <= radius:
            pts.append(p)
            if len(pts) >= n_pts:
                break
    pts = np.array(pts, dtype=float)
    if len(pts) > 0:
        pts *= (radius / (np.max(np.linalg.norm(pts, axis=1)) + 1e-9))
    return pts

def embed_quasicrystal_layers(q2d, n_layers=6, layer_spacing=2.2, twist=0.33):
    q2d = np.asarray(q2d)
    pts3, meta = [], []
    for L in range(n_layers):
        theta = L * twist
        R = np.array([[math.cos(theta), -math.sin(theta)],
                      [math.sin(theta),  math.cos(theta)]], dtype=float)
        layer = (q2d @ R.T)
        z = (L - (n_layers-1)/2.0) * layer_spacing
        for i, (x,y) in enumerate(layer):
            pts3.append([x, y, z])
            meta.append({"layer": L, "qid": i})
    return np.array(pts3, dtype=float), meta

def build_quasicrystal_chain_graph(points3, meta, within_layer_k=10, cross_layer_k=4, cross_sigma=2.8):
    P = np.asarray(points3)
    N = P.shape[0]
    Gq = nx.Graph()
    for i in range(N):
        Gq.add_node(i, pos=P[i], **meta[i])

    layers = {}
    for i, m in enumerate(meta):
        layers.setdefault(m["layer"], []).append(i)

    for L, idxs in layers.items():
        if len(idxs) < 2:
            continue
        Pl = P[idxs]
        knn = k_nearest(Pl, k=min(within_layer_k, len(idxs)-1))
        for li, neigh in enumerate(knn):
            i = idxs[li]
            for lj in neigh:
                j = idxs[lj]
                d = float(np.linalg.norm(P[i] - P[j]))
                Gq.add_edge(i, j, kind="quasi_local", w=1.0/(d+1e-9), dist=d)

    layer_list = sorted(layers.keys())
    for L in layer_list:
        if (L+1) not in layers:
            continue
        A = layers[L]
        B = layers[L+1]
        PA = P[A]
        PB = P[B]
        for ia, i in enumerate(A):
            xy = PA[ia][:2]
            dxy = np.sum((PB[:, :2] - xy)**2, axis=1)
            nn = np.argsort(dxy)[:min(cross_layer_k, len(B))]
            for bidx in nn:
                j = B[int(bidx)]
                d = float(np.linalg.norm(P[i] - P[j]))
                p = gaussian_prob(math.sqrt(float(dxy[bidx])), sigma=cross_sigma)
                if random.random() < p:
                    Gq.add_edge(i, j, kind="quasi_chain", w=1.0/(d+1e-9), dist=d)
    return Gq

# ============================================================
# 3) Macro regions + wiring
# ============================================================

REGIONS = [
    ("OlfactoryBulb", np.array([0.0,  0.0,  10.0]), 0.17),
    ("Cortex",        np.array([0.0,  0.0,   4.0]), 0.34),
    ("Hippocampus",   np.array([0.0, -5.0,   1.5]), 0.13),
    ("Thalamus",      np.array([0.0,  0.0,   0.0]), 0.13),
    ("Cerebellum",    np.array([0.0, -7.0,  -6.0]), 0.16),
    ("Brainstem",     np.array([0.0,  0.0,  -9.0]), 0.07),
]

def assign_region(pos3):
    p = np.asarray(pos3)
    best, bestd = None, 1e18
    for name, c, _ in REGIONS:
        d = float(np.linalg.norm(p - c))
        if d < bestd:
            bestd = d
            best = name
    return best

def add_macro_wiring(G, local_sigma=1.7, long_sigma=7.0,
                     local_p=0.22, long_p=0.012, commissure_p=0.05):
    nodes = list(G.nodes())
    P = np.array([G.nodes[n]["pos"] for n in nodes], dtype=float)
    region_of = {n: G.nodes[n]["region"] for n in nodes}
    N = len(nodes)

    knn_local = k_nearest(P, k=min(18, N-1)) if N > 1 else []

    for i in range(N):
        ni = nodes[i]
        ri = region_of[ni]
        for j in knn_local[i]:
            nj = nodes[j]
            rj = region_of[nj]
            d = float(np.linalg.norm(P[i] - P[j]))
            if ri == rj:
                p = local_p * gaussian_prob(d, local_sigma)
            else:
                p = 0.35 * local_p * gaussian_prob(d, local_sigma)
            if random.random() < p:
                G.add_edge(ni, nj, kind="macro_local", w=1.0/(d+1e-9), dist=d)

    trials = int(N * 18)
    for _ in range(trials):
        i = random.randrange(N)
        j = random.randrange(N)
        if i == j:
            continue
        ni, nj = nodes[i], nodes[j]
        if region_of[ni] == region_of[nj]:
            continue
        d = float(np.linalg.norm(P[i] - P[j]))
        p = long_p * gaussian_prob(d, long_sigma)
        if random.random() < p:
            G.add_edge(ni, nj, kind="macro_long", w=1.0/(d+1e-9), dist=d)

    for i in range(N):
        ni = nodes[i]
        p = P[i].copy()
        target = np.array([-p[0], p[1], p[2]])
        d2 = np.sum((P - target)**2, axis=1)
        j = int(np.argmin(d2))
        if i != j:
            nj = nodes[j]
            d = float(np.linalg.norm(P[i] - P[j]))
            if random.random() < commissure_p * gaussian_prob(d, long_sigma):
                G.add_edge(ni, nj, kind="commissure", w=1.0/(d+1e-9), dist=d)

# ============================================================
# 4) Fuse core + quasi and build v2 scaffold
# ============================================================

def fuse_graphs(G_core, G_quasi, stitch_k=6, stitch_thresh=2.3):
    core_nodes = list(G_core.nodes())
    quasi_nodes = list(G_quasi.nodes())
    core_pos = np.array([G_core.nodes[i]["pos"] for i in core_nodes], dtype=float)
    quasi_pos = np.array([G_quasi.nodes[i]["pos"] for i in quasi_nodes], dtype=float)

    H = nx.Graph()
    for i in core_nodes:
        H.add_node(("core", i), **G_core.nodes[i], family="core")
    for i in quasi_nodes:
        H.add_node(("quasi", i), **G_quasi.nodes[i], family="quasi")

    for (u,v,data) in G_core.edges(data=True):
        H.add_edge(("core", u), ("core", v), **data)
    for (u,v,data) in G_quasi.edges(data=True):
        H.add_edge(("quasi", u), ("quasi", v), **data)

    for qi, qpos in enumerate(quasi_pos):
        d2 = np.sum((core_pos - qpos)**2, axis=1)
        nn = np.argsort(d2)[:min(stitch_k, len(core_nodes))]
        for idx in nn:
            d = float(math.sqrt(d2[int(idx)]))
            if d <= stitch_thresh:
                cu = ("quasi", quasi_nodes[qi])
                cv = ("core", core_nodes[int(idx)])
                H.add_edge(cu, cv, kind="stitch", w=1.0/(d+1e-9), dist=d)
    return H

def build_mouse_brain_scaffold(seed=7, fcc_bounds=10.5, fcc_a=1.0, quasi_pts=520, quasi_layers=7, layer_spacing=2.2, twist=0.31):
    set_seed(seed)

    core_pts = generate_fcc_points(bounds=fcc_bounds, a=fcc_a)
    core_pts = core_pts.copy()
    core_pts[:, 2] *= 1.25
    Gc = build_dodecahedral_graph(core_pts, nn_k=14, shell_factor=1.18)

    q2d = ammann_beenker_points_2d(n_pts=quasi_pts, seed=seed, radius=14.0)
    q3, meta = embed_quasicrystal_layers(q2d, n_layers=quasi_layers, layer_spacing=layer_spacing, twist=twist)

    R = rot_matrix([1, 0, 0], theta=0.55) @ rot_matrix([0, 0, 1], theta=0.35)
    q3 = (q3 @ R.T)
    Gq = build_quasicrystal_chain_graph(q3, meta, within_layer_k=10, cross_layer_k=4)

    H = fuse_graphs(Gc, Gq, stitch_k=6, stitch_thresh=2.3)

    for n in H.nodes():
        H.nodes[n]["region"] = assign_region(H.nodes[n]["pos"])

    add_macro_wiring(H, local_sigma=1.7, long_sigma=7.0, local_p=0.22, long_p=0.012, commissure_p=0.05)
    return H

def graph_report(G: nx.Graph):
    nodes = G.number_of_nodes()
    edges = G.number_of_edges()
    comps = list(nx.connected_components(G))
    comps_sorted = sorted([len(c) for c in comps], reverse=True)
    giant = comps_sorted[0] if comps_sorted else 0
    avg_deg = (2*edges / nodes) if nodes else 0.0

    kinds = {}
    for _, _, d in G.edges(data=True):
        k = d.get("kind", "unknown")
        kinds[k] = kinds.get(k, 0) + 1

    regions = {}
    for _, d in G.nodes(data=True):
        r = d.get("region", "unknown")
        regions[r] = regions.get(r, 0) + 1

    return {
        "nodes": nodes,
        "edges": edges,
        "avg_degree": avg_deg,
        "components": len(comps),
        "giant_component": giant,
        "edge_kinds": dict(sorted(kinds.items(), key=lambda x: -x[1])),
        "regions": dict(sorted(regions.items(), key=lambda x: -x[1])),
    }

# ============================================================
# 5) GEXF-safe export (FIX)
# ============================================================

def _sanitize_for_gexf(G):
    H = nx.Graph()
    for n, d in G.nodes(data=True):
        dd = dict(d)
        p = dd.get("pos", None)
        if isinstance(p, np.ndarray):
            dd["x"] = float(p[0]); dd["y"] = float(p[1]); dd["z"] = float(p[2])
            dd.pop("pos", None)
        elif isinstance(p, (list, tuple)) and len(p) == 3:
            dd["x"] = float(p[0]); dd["y"] = float(p[1]); dd["z"] = float(p[2])
            dd.pop("pos", None)
        for k, v in list(dd.items()):
            if isinstance(v, np.ndarray):
                dd[k] = str(v.tolist())
            elif isinstance(v, (list, tuple, dict)):
                dd[k] = str(v)
        H.add_node(n, **dd)

    for u, v, d in G.edges(data=True):
        dd = dict(d)
        for k, val in list(dd.items()):
            if isinstance(val, np.ndarray):
                dd[k] = str(val.tolist())
            elif isinstance(val, (list, tuple, dict)):
                dd[k] = str(val)
            elif isinstance(val, (np.floating, np.integer)):
                dd[k] = float(val)
        H.add_edge(u, v, **dd)
    return H

def save_graph_artifacts(G, out_dir):
    os.makedirs(out_dir, exist_ok=True)

    gexf_path = os.path.join(out_dir, "mousebrain_v2_scaffold.gexf")
    H = _sanitize_for_gexf(G)
    nx.write_gexf(H, gexf_path)

    nodes = []
    for n, d in G.nodes(data=True):
        p = d.get("pos", [0,0,0])
        if isinstance(p, np.ndarray):
            p = p.tolist()
        nodes.append({
            "id": str(n),
            "x": float(p[0]), "y": float(p[1]), "z": float(p[2]),
            "region": d.get("region",""),
            "family": d.get("family",""),
        })
    edges = []
    for u, v, d in G.edges(data=True):
        edges.append({
            "u": str(u), "v": str(v),
            "kind": d.get("kind",""),
            "dist": float(d.get("dist", 0.0)),
            "w": float(d.get("w", 1.0)),
        })
    with open(os.path.join(out_dir, "nodes.json"), "w") as f:
        json.dump(nodes, f, indent=2)
    with open(os.path.join(out_dir, "edges.json"), "w") as f:
        json.dump(edges, f, indent=2)

    return gexf_path

# ============================================================
# 6) v2 Dynamics: LIF + delays + STDP
# ============================================================

class MouseBrainV2Sim:
    def __init__(self, G: nx.Graph, seed=7):
        set_seed(seed)
        self.G = G.copy()
        self.nodes = list(self.G.nodes())
        self.N = len(self.nodes)
        self.pos = np.array([self.G.nodes[n]["pos"] for n in self.nodes], dtype=float)
        self.region = [self.G.nodes[n]["region"] for n in self.nodes]
        self.family = [self.G.nodes[n].get("family","core") for n in self.nodes]

        self.idx = {n:i for i,n in enumerate(self.nodes)}
        self.edges = []
        for u,v,d in self.G.edges(data=True):
            iu, iv = self.idx[u], self.idx[v]
            self.edges.append((iu, iv, d))

    def init_neurons(self, p_inhib=0.18):
        self.is_inhib = (np.random.rand(self.N) < p_inhib)
        region_boost = {
            "OlfactoryBulb": 1.10,
            "Cortex": 1.00,
            "Hippocampus": 1.05,
            "Thalamus": 0.95,
            "Cerebellum": 0.90,
            "Brainstem": 0.98
        }
        self.excite_gain = np.array([region_boost.get(r,1.0) for r in self.region], dtype=float)

        self.V = np.zeros(self.N, dtype=float)
        self.V_rest = -0.065
        self.V_reset = -0.070
        self.V_th = -0.050
        self.tau_m = 0.020
        self.refractory = np.zeros(self.N, dtype=int)

        base = 0.35
        self.I_bg = base * self.excite_gain * (0.8 + 0.4*np.random.rand(self.N))

        self.spikes = []

    def init_synapses(self, w_init=0.35, w_inhib=-0.55, plastic_frac=0.55, max_out=18):
        self.syn_u = []
        self.syn_v = []
        self.syn_w = []
        self.syn_plastic = []
        self.syn_delay = []

        speed = 5.8
        out_counts = np.zeros(self.N, dtype=int)
        e = self.edges[:]
        random.shuffle(e)

        for iu, iv, d in e:
            dist = float(d.get("dist", np.linalg.norm(self.pos[iu]-self.pos[iv])))
            delay_ms = max(1.0, min(12.0, dist / speed))
            delay_steps = int(round(delay_ms))

            for (a,b) in [(iu,iv),(iv,iu)]:
                if out_counts[a] >= max_out:
                    continue
                out_counts[a] += 1
                self.syn_u.append(a)
                self.syn_v.append(b)
                if self.is_inhib[a]:
                    w = w_inhib * (0.9 + 0.2*np.random.rand())
                    plastic = False
                else:
                    w = w_init * (0.7 + 0.6*np.random.rand())
                    plastic = (np.random.rand() < plastic_frac)
                self.syn_w.append(w)
                self.syn_plastic.append(plastic)
                self.syn_delay.append(delay_steps)

        self.syn_u = np.array(self.syn_u, dtype=int)
        self.syn_v = np.array(self.syn_v, dtype=int)
        self.syn_w = np.array(self.syn_w, dtype=float)
        self.syn_plastic = np.array(self.syn_plastic, dtype=bool)
        self.syn_delay = np.array(self.syn_delay, dtype=int)

        self.M = len(self.syn_u)

        self.tau_pre = 0.020
        self.tau_post = 0.020
        self.A_plus = 0.012
        self.A_minus = 0.014
        self.w_min = 0.05
        self.w_max = 1.20

        self.pre_trace = np.zeros(self.N, dtype=float)
        self.post_trace = np.zeros(self.N, dtype=float)

        self.max_delay = int(np.max(self.syn_delay)) if self.M else 1
        self.queue = [[] for _ in range(self.max_delay + 1)]

    def run(self, T_ms=2000, dt_ms=1.0, p_external=0.005):
        dt = dt_ms / 1000.0
        self.V[:] = self.V_rest + 0.005*np.random.randn(self.N)
        I_syn = np.zeros(self.N, dtype=float)

        out_lists = [[] for _ in range(self.N)]
        for s in range(self.M):
            out_lists[self.syn_u[s]].append(s)

        decay_pre = math.exp(-dt / self.tau_pre)
        decay_post = math.exp(-dt / self.tau_post)

        for t in range(int(T_ms)):
            self.pre_trace *= decay_pre
            self.post_trace *= decay_post

            bucket = self.queue[t % (self.max_delay + 1)]
            if bucket:
                for s in bucket:
                    post = self.syn_v[s]
                    I_syn[post] += self.syn_w[s]
                bucket.clear()

            I = self.I_bg + I_syn
            ext_spikers = np.where(np.random.rand(self.N) < p_external)[0]
            if len(ext_spikers) > 0:
                I[ext_spikers] += 0.6 * (0.8 + 0.4*np.random.rand(len(ext_spikers)))

            I_syn[:] = 0.0

            active = self.refractory <= 0
            self.refractory[~active] -= 1

            dV = ((self.V_rest - self.V) + 0.020*I) / self.tau_m
            self.V[active] += dt * dV[active]

            spk = np.where((self.V >= self.V_th) & active)[0]
            if len(spk) > 0:
                for i in spk.tolist():
                    self.spikes.append((t, int(i)))

                self.pre_trace[spk] += 1.0
                self.post_trace[spk] += 1.0

                for pre in spk.tolist():
                    for s in out_lists[pre]:
                        dly = self.syn_delay[s]
                        self.queue[(t + dly) % (self.max_delay + 1)].append(s)

                for pre in spk.tolist():
                    outs = out_lists[pre]
                    if outs:
                        outs = np.array(outs, dtype=int)
                        plastic_mask = self.syn_plastic[outs]
                        if np.any(plastic_mask):
                            outs_p = outs[plastic_mask]
                            posts = self.syn_v[outs_p]
                            dw = self.A_plus * self.post_trace[posts]
                            self.syn_w[outs_p] = np.clip(self.syn_w[outs_p] + dw, self.w_min, self.w_max)

                for post in spk.tolist():
                    inc = np.where(self.syn_v == post)[0]
                    if inc.size > 0:
                        plastic_mask = self.syn_plastic[inc]
                        if np.any(plastic_mask):
                            inc_p = inc[plastic_mask]
                            pres = self.syn_u[inc_p]
                            dw = self.A_minus * self.pre_trace[pres]
                            self.syn_w[inc_p] = np.clip(self.syn_w[inc_p] - dw, self.w_min, self.w_max)

                self.V[spk] = self.V_reset
                self.refractory[spk] = 2

        return {"T_ms": int(T_ms), "N": int(self.N), "M": int(self.M), "spike_count": int(len(self.spikes))}

    def summarize(self):
        if not self.spikes:
            return {"firing_rate_hz_mean": 0.0, "active_frac": 0.0}
        T = max(t for t,_ in self.spikes) + 1
        counts = np.zeros(self.N, dtype=int)
        for t,i in self.spikes:
            counts[i] += 1
        fr = counts / (T/1000.0 + 1e-12)
        return {
            "T_ms": int(T),
            "firing_rate_hz_mean": float(np.mean(fr)),
            "firing_rate_hz_p95": float(np.percentile(fr, 95)),
            "active_frac": float(np.mean(fr > 0.1)),
            "syn_w_mean": float(np.mean(self.syn_w)) if self.M else 0.0,
            "syn_w_p95": float(np.percentile(self.syn_w, 95)) if self.M else 0.0,
            "syn_w_min": float(np.min(self.syn_w)) if self.M else 0.0,
            "syn_w_max": float(np.max(self.syn_w)) if self.M else 0.0,
        }

# ============================================================
# 7) Output helpers
# ============================================================

def plot_3d_scaffold(G, out_png, max_edges=8000):
    nodes = list(G.nodes())
    pos = np.array([G.nodes[n]["pos"] for n in nodes], dtype=float)
    fam = np.array([0 if G.nodes[n].get("family") == "core" else 1 for n in nodes], dtype=int)

    edges = list(G.edges(data=True))
    if len(edges) > max_edges:
        edges = random.sample(edges, max_edges)

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    core_idx = np.where(fam == 0)[0]
    quasi_idx = np.where(fam == 1)[0]

    ax.scatter(pos[core_idx,0], pos[core_idx,1], pos[core_idx,2], s=5, alpha=0.55, marker="o")
    ax.scatter(pos[quasi_idx,0], pos[quasi_idx,1], pos[quasi_idx,2], s=5, alpha=0.55, marker="^")

    for (u,v,d) in edges:
        pu = G.nodes[u]["pos"]
        pv = G.nodes[v]["pos"]
        ax.plot([pu[0], pv[0]], [pu[1], pv[1]], [pu[2], pv[2]], alpha=0.07)

    ax.set_title("MouseBrain v2 Scaffold (Core + Quasi Tracts)")
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    plt.tight_layout()
    fig.savefig(out_png, dpi=170)
    plt.close(fig)

def plot_spike_raster(spikes, out_png, max_points=120000):
    if not spikes:
        fig = plt.figure(figsize=(10,4))
        plt.title("Spike raster (no spikes)")
        plt.tight_layout()
        fig.savefig(out_png, dpi=170)
        plt.close(fig)
        return
    spikes = spikes if len(spikes) <= max_points else random.sample(spikes, max_points)
    t = np.array([s[0] for s in spikes], dtype=int)
    i = np.array([s[1] for s in spikes], dtype=int)
    fig = plt.figure(figsize=(11,4))
    plt.scatter(t, i, s=1, alpha=0.35)
    plt.xlabel("time (ms)")
    plt.ylabel("neuron index")
    plt.title("Spike raster (sampled)")
    plt.tight_layout()
    fig.savefig(out_png, dpi=170)
    plt.close(fig)

def plot_weight_hist(weights, out_png):
    fig = plt.figure(figsize=(8,4))
    if len(weights) > 0:
        plt.hist(weights, bins=60, alpha=0.85)
        plt.xlabel("synaptic weight")
        plt.ylabel("count")
        plt.title("Final synaptic weight histogram")
    else:
        plt.title("Final synaptic weight histogram (no synapses)")
    plt.tight_layout()
    fig.savefig(out_png, dpi=170)
    plt.close(fig)

def bundle_zip(out_dir, zip_path):
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(out_dir):
            for fn in files:
                fp = os.path.join(root, fn)
                arc = os.path.relpath(fp, out_dir)
                z.write(fp, arcname=arc)
    return zip_path

# ============================================================
# 8) FULL AUTO RUN
# ============================================================

CFG = {
    "seed": 7,
    "fcc_bounds": 10.5,
    "fcc_a": 1.0,
    "quasi_pts": 520,
    "quasi_layers": 7,
    "layer_spacing": 2.2,
    "twist": 0.31,
    "sim_T_ms": 2500,
    "p_external": 0.006,
}

ts = time.time()
run_id = time.strftime("%Y%m%d_%H%M%S")
OUT = os.path.join(os.getcwd(), f"out_mousebrain_v2_{run_id}")
os.makedirs(OUT, exist_ok=True)

print("OUT:", OUT)
print("Building scaffold...")
G = build_mouse_brain_scaffold(
    seed=CFG["seed"],
    fcc_bounds=CFG["fcc_bounds"],
    fcc_a=CFG["fcc_a"],
    quasi_pts=CFG["quasi_pts"],
    quasi_layers=CFG["quasi_layers"],
    layer_spacing=CFG["layer_spacing"],
    twist=CFG["twist"],
)

rep = graph_report(G)
with open(os.path.join(OUT, "scaffold_report.json"), "w") as f:
    json.dump(rep, f, indent=2)

print("Scaffold report:", rep)

print("Saving graph artifacts (GEXF-safe)...")
gexf_path = save_graph_artifacts(G, OUT)
print("GEXF:", gexf_path)

print("Plotting scaffold...")
plot_3d_scaffold(G, os.path.join(OUT, "scaffold_3d.png"), max_edges=9000)

print("Initializing simulation...")
sim = MouseBrainV2Sim(G, seed=CFG["seed"])
sim.init_neurons(p_inhib=0.18)
sim.init_synapses(w_init=0.35, w_inhib=-0.55, plastic_frac=0.55, max_out=18)

print("Running simulation...")
sim_meta = sim.run(T_ms=CFG["sim_T_ms"], dt_ms=1.0, p_external=CFG["p_external"])
sim_sum = sim.summarize()

with open(os.path.join(OUT, "sim_meta.json"), "w") as f:
    json.dump(sim_meta, f, indent=2)
with open(os.path.join(OUT, "sim_summary.json"), "w") as f:
    json.dump(sim_sum, f, indent=2)

spikes = np.array(sim.spikes, dtype=int) if sim.spikes else np.zeros((0,2), dtype=int)
np.save(os.path.join(OUT, "spikes.npy"), spikes)
np.save(os.path.join(OUT, "weights_final.npy"), sim.syn_w)

print("Plotting spikes + weights...")
plot_spike_raster(sim.spikes, os.path.join(OUT, "spike_raster.png"))
plot_weight_hist(sim.syn_w, os.path.join(OUT, "weights_hist.png"))

syn_table = {
    "u": sim.syn_u.tolist(),
    "v": sim.syn_v.tolist(),
    "w": sim.syn_w.tolist(),
    "delay_ms": sim.syn_delay.tolist(),
    "plastic": sim.syn_plastic.tolist(),
}
with open(os.path.join(OUT, "synapses_directed.json"), "w") as f:
    json.dump(syn_table, f)

ZIP = os.path.join(os.getcwd(), f"MouseBrain_ODQCL_v2_Bundle_{run_id}.zip")
bundle_zip(OUT, ZIP)

elapsed = time.time() - ts
print("\\nDONE")
print("Elapsed (s):", round(elapsed, 2))
print("Bundle:", ZIP)
print("Sim summary:", sim_sum)


In [ ]:
# Convenience: list latest bundles
import glob
sorted(glob.glob("MouseBrain_ODQCL_v2_Bundle_*.zip"))[-5:]
